# Importación de librerías

In [1]:
from pathlib import Path

from langchain_core.documents.base import Document
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings

e:\proyecto\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Importación de datos

Recogemos todo el corpus de la carpeta `processed/corpus`.

In [2]:
ROOT_DIR = Path("E:/proyecto")
CORPUS_DIR = ROOT_DIR / "data" / "processed" / "corpus"

docs = []
for doc_path in sorted(CORPUS_DIR.rglob("*.md")):
    species = doc_path.relative_to(CORPUS_DIR).parts[0]

    with open(doc_path, 'r', encoding='utf-8') as f:
        text = f.read()

    doc = Document(
        page_content=text,
        metadata = {
            "species_name": species.replace("_", " ").capitalize() # Lo dejamos con el nombre "correcto" de la planta para facilitar la búsqueda de datos.
        }
    )

    docs.append(doc)

print(f"Importados {len(docs)} documentos.")

Importados 91 documentos.


Realizamos la división en chunks de los datos utilizando `RecursiveCharacterTextSplitter`. En este caso, creamos divisiones de 500 caracteres con 100 caracteres de overlap.

In [3]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 100
)

chunked_docs = splitter.split_documents(docs)
print(f"Dividido en {len(chunked_docs)} chunks.")

Dividido en 527 chunks.


# Creación de base de datos vectorial

Cargamos nuestro modelo para realizar los embeddings.

In [4]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-qwen3-embedding-0.6b",
    base_url="http://localhost:1234/v1",
    api_key="lm-studio-dummy",
    check_embedding_ctx_length=False
)

print("Modelo de embeddings cargado.")

Modelo de embeddings cargado.


Y creamos nuestra base de datos vectorial utilizando Chroma.

In [5]:
vector_store = Chroma.from_documents(
    documents = chunked_docs,
    embedding = embeddings,
    persist_directory = str(ROOT_DIR / "chroma_db")
)

print("Base de datos de vectores creada correctamente.")

Base de datos de vectores creada correctamente.


# Probando el RAG

In [8]:
vector_store.similarity_search_with_score(
    "Háblame de la distribución del dracena draco",
    k=10,
    filter={"species_name": "Dracaena draco"}
)

[(Document(id='8fe22a71-a080-49d5-aa4f-944161e8d798', metadata={'species_name': 'Dracaena draco'}, page_content='## Descripción'),
  0.8305777311325073),
 (Document(id='179bf24c-574e-4a0f-8280-6b9f7419ddeb', metadata={'species_name': 'Dracaena draco'}, page_content='# Dracaena draco  L. subsp.  draco\n\n## Nombre común:\n\ndrago\n\n.\n\n## Familia:\n\nDracaenaceae.\n\n## Floración:\n\ndiciembre-septiembre.\n\nDistribución archipiélagos:\n\nAzo Mad Can CV, Islas: L C - M P - H P G T C - A V N T F R\n\n## Características:'),
  0.9449803829193115),
 (Document(id='f2d45cae-745a-4419-a5e5-00a966fc2276', metadata={'species_name': 'Dracaena draco'}, page_content='# Drago\n\nNombre común: *Drago* Nombre científico: *Dracaena draco L.* http//:www.biodiversidadcanarias.es \n\nNombre común: *Drago de Gran Canaria* Nombre científico: *Dracaena tamaranae Marrero Rodr., Almeida-Pérez & González-Martín* http//:www.biodiversidadcanarias.es'),
  1.0080125331878662),
 (Document(id='d697aa95-c8d5-4090-8e